# NIH metrics

This notebook uses information extracted from [NIH Exporter](https://reporter.nih.gov/exporter) to identify NIH-funded users of PhysioNet.

## Import packages

In [92]:
import os
from pathlib import Path

import pandas as pd

from twentyfiveyears.nih import (combine_exporter_tables, get_physionet_users, get_investigators, get_authors, link_users)

In [107]:
# Now, if you make changes to the 'twentyfiveyears.nih' module, you can reload it like this:  
import importlib  
import twentyfiveyears.nih  # Import the module itself  
  
# Reload the module to apply any changes  
importlib.reload(twentyfiveyears.nih)  
  
# Re-import the specific functions you need  
from twentyfiveyears.nih import (combine_exporter_tables, get_physionet_users, get_investigators, get_authors, link_users)

## Setup with Synthetic Data

In [94]:
# Create synthetic person map
synthetic_lists = [list(range(100000000, 100000009)), list(range(1, 10))]
df_synthetic_person_map = pd.DataFrame(synthetic_lists).transpose()
df_synthetic_person_map.columns=['person_id', 'physionet_id']

In [95]:
test_path = os.path.join("..", "tests", "data", "synthetic_names")

In [96]:
# Get the synthetic names for PhysioNet and NIH
df_synthetic_physionet_users = get_physionet_users(os.path.join(test_path,'physionet_users.csv'), df_synthetic_person_map, first_name_as_initial=True)
df_synthetic_physionet_users.head()

,person_id,physionet_name
0,100000000,t rollins
1,100000001,f mendes de sarria
2,100000002,j smith
3,100000003,a swan
4,100000004,j henderson


In [97]:
synthetic_investigators = get_investigators(pd.read_csv(os.path.join(test_path, "nih_investigators.csv")), first_name_as_initial=True)
synthetic_investigators[0:]

['j smith',
 'm sanchez',
 'm jones',
 'j wilson',
 'r carey',
 'j henderson',
 'm winslow le heritage',
 'a kelsey',
 'b lopez',
 'm jackson',
 'a macenroe',
 's taylor',
 's moore-row']

In [98]:
synthetic_authors = get_authors(pd.read_csv(os.path.join(test_path, "nih_authors.csv")), first_name_as_initial=True)
synthetic_authors[0:]

['l nelson',
 'b oxford',
 'f mendes de sarria',
 'y carpenter',
 'r mandez',
 'a swan',
 'r jones',
 'j henderson',
 'k williams-garcia',
 'd miller-so',
 's mccarthy',
 'j kelsey',
 'j swift',
 'm henri',
 'k appleton',
 's hope',
 's brown',
 "s o'leary",
 's jackson',
 't virginia']

## Link Synthetic Users

In [99]:
limit = None
df_physionet_users = link_users(df_synthetic_physionet_users, synthetic_investigators, match_group="investigators", limit=limit)

100%|██████████| 9/9 [00:00<00:00, 1515.16it/s]

Finished in 0.04056191444396973 seconds


In [100]:
df_physionet_users = link_users(df_synthetic_physionet_users, synthetic_authors, match_group="authors", limit=limit)

100%|██████████| 9/9 [00:00<00:00, 2035.74it/s]

Finished in 0.04941082000732422 seconds


In [101]:
df_physionet_users.head(10)

,person_id,physionet_name,matched_investigator_score,matched_investigator_name,matched_author_score,matched_author_name
0,100000000,t rollins,0.694444,j wilson,0.689947,r jones
1,100000001,f mendes de sarria,0.627465,j henderson,1.000000,f mendes de sarria
2,100000002,j smith,1.000000,j smith,0.866667,j swift
3,100000003,a swan,0.688889,a macenroe,1.000000,a swan
4,100000004,j henderson,1.000000,j henderson,1.000000,j henderson
5,100000005,a jones,0.904762,m jones,0.904762,r jones
6,100000006,k williams-garcia,0.639706,j wilson,1.000000,k williams-garcia
7,100000007,m davis,0.671958,m jackson,0.656277,d miller-so
8,100000008,b johnson brown,0.682540,m jones,0.682540,r jones


## Setup with Actual Data

In [102]:
# Set the base path
base_path = os.path.join("..", "data")

## Load map of Person IDs

All users are assigned a unique `person_id`.

In [103]:
# Load the map of Person IDs
path = os.path.join(base_path, 'handcrafted', 'person_id_lookup.csv')
person_map = pd.read_csv(path)
person_map.head(3)

,person_id,physionet_id
0,100000000,2
1,100000001,6
2,100000002,8


## Load PhysioNet dataset

Load a dataset containing the list of PhysioNet users

In [104]:
# Load DataFrame of PhysioNet users
path = os.path.join(base_path, 'physionet', 'users.csv')
physionet_users = get_physionet_users(path, person_map, first_name_as_initial=True)
physionet_users = physionet_users[0:9]
physionet_users.head(3)

,person_id,physionet_name
0,100000000,f torres fábregas
1,100000001,t pollard
2,100000002,b moody


## Load Principal Investigators of NIH projects

Load a list of Principal Investigators

In [105]:
# Load the NIH project data
path = os.path.join(base_path, 'nih', 'exporter', 'projects')
projects = combine_exporter_tables(path, "RePORTER_PRJ_C_FY", start_year=1995)
projects.head(3)

,APPLICATION_ID,ACTIVITY,ADMINISTERING_IC,APPLICATION_TYPE,ARRA_FUNDED,AWARD_NOTICE_DATE,BUDGET_START,BUDGET_END,CFDA_CODE,CORE_PROJECT_NUM,...,SUBPROJECT_ID,SUFFIX,SUPPORT_YEAR,TOTAL_COST,TOTAL_COST_SUB_PROJECT,OPPORTUNITY NUMBER,FUNDING_MECHANISM,ORG_IPF_CODE,DIRECT_COST_AMT,INDIRECT_COST_AMT
0,2056372,A03,AH,1,NaN,1995-05-19T00:00:00,07/01/1995,06/30/1996,NaN,A03AH001117,...,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2056373,A03,AH,1,NaN,1995-05-19T00:00:00,07/01/1995,06/30/1996,NaN,A03AH001118,...,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2056374,A03,AH,1,NaN,1995-05-19T00:00:00,07/01/1995,06/30/1996,NaN,A03AH001119,...,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [108]:
# Get the names of Principal Investigators
investigators = get_investigators(projects, first_name_as_initial=True)
investigators[0:3]

['e capilouto', 'a afifi', 'r hart']

## Load authors of publications linked to NIH projects

Load a list of authors linked to NIH projects

In [109]:
# Load the NIH publications data
path = os.path.join(base_path, 'nih', 'exporter', 'publications')
publications = combine_exporter_tables(path, "RePORTER_PUB_C_", start_year=1995)
publications.head(3)

,AFFILIATION,AUTHOR_LIST,COUNTRY,ISSN,JOURNAL_ISSUE,JOURNAL_TITLE,JOURNAL_TITLE_ABBR,JOURNAL_VOLUME,LANG,PAGE_NUMBER,PMC_ID,PMID,PUB_DATE,PUB_TITLE,PUB_YEAR
0,"Department of Fisheries and Wildlife, Oregon S...","Curtis, L R; Zhang, Q; el-Zahr, C; Carpenter, ...",UNITED STATES,0272-0590,1,Fundamental and applied toxicology : official...,Fundam Appl Toxicol,25,eng,146-53,NaN,7601322,1995 Apr,Temperature-modulated incidence of aflatoxin B...,1995
1,"Department of Biology, Boston University, Mass...","Loechler, E L",UNITED STATES,0899-1987,4,Molecular carcinogenesis.,Mol Carcinog,13,eng,213-9,NaN,7646760,1995 Aug,How are potent bulky carcinogens able to induc...,1995
2,Department of Pharmacology and Toxicology Scho...,"Carlson, G P; Olson, R M",AUSTRALIA,1039-9712,1,Biochemistry and molecular biology internation...,Biochem Mol Biol Int,37,eng,65-71,NaN,8653089,1995 Sep,Comparison of the metabolism of alcohols by ra...,1995


In [110]:
# Get the names of authors
authors = get_authors(publications, first_name_as_initial=True)
authors[0:3]

['l curtis', 'q zhang', 'c el-zahr']

## Match NIH listed people to PhysioNet users

Attempt to match people between the two sources

In [111]:
# Match NIH Principal Investigators to PhysioNet users
# Set limit for testing
limit = None
physionet_users = link_users(physionet_users, investigators, match_group="investigators", limit=limit)

100%|██████████| 9/9 [00:00<00:00, 28.69it/s]


Finished in 7.130033016204834 seconds


In [112]:
# Match NIH authors to PhysioNet users
physionet_users = link_users(physionet_users, authors, match_group="authors", limit=limit)

100%|██████████| 9/9 [00:00<00:00, 3188.24it/s]


Finished in 4.607928991317749 seconds


In [113]:
physionet_users.head(5)

,person_id,physionet_name,matched_investigator_score,matched_investigator_name,matched_author_score,matched_author_name
0,100000000,f torres fábregas,0.894118,f torres,0.894118,f torres
1,100000001,t pollard,1.000000,t pollard,1.000000,t pollard
2,100000002,b moody,0.921429,b modney,1.000000,b moody
3,100000003,a johnson,1.000000,a johnson,1.000000,a johnson
4,100000004,j ishii-rousseau,0.862500,j ishida,0.887500,j ishii


Save the results

## Save the results

In [159]:
# Save the results
save_path = os.path.join(base_path, 'physionet_users_nih_funded.csv')
path = Path(save_path)

# Convert to a path that works on the current OS
normalized_path = path.as_posix() if path.drive else Path(*path.parts).resolve()

# Output the merged DataFrame or save it to a file
physionet_users.to_csv(normalized_path, index=False)